In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib
from datetime import datetime

# 1. GENERATE DUMMY DATA (Replace this with your MongoDB data later)
# We are simulating realistic data based on typical Sri Lankan rental prices
np.random.seed(42)
n_samples = 1000

data = {
    'Bike_Type': np.random.choice(['Scooter', 'Gear Bike'], n_samples),
    'Engine_CC': np.random.choice([110, 125, 150, 250], n_samples),
    'Mfg_Year': np.random.randint(2015, 2026, n_samples),
    'Fuel_Type': np.random.choice(['Petrol', 'Electric'], n_samples, p=[0.8, 0.2]),
    'City': np.random.choice(['Galle', 'Mirissa', 'Deniyaya', 'Thalpe', 'Colombo'], n_samples),
    'Is_Peak_Season': np.random.choice([0, 1], n_samples)
}

df = pd.DataFrame(data)

# Simulate target variable (Price Per Day in LKR) based on logical rules
# Base price around 1500 LKR
base_price = 1500
df['Price_LKR'] = base_price + \
                  (df['Engine_CC'] * 5) + \
                  (df['Is_Peak_Season'] * 800) + \
                  (np.where(df['Bike_Type'] == 'Gear Bike', 500, 0)) - \
                  ((2026 - df['Mfg_Year']) * 100) # Older bikes are cheaper

# Add some random noise to simulate real-world price variations
df['Price_LKR'] += np.random.normal(0, 200, n_samples)
df['Price_LKR'] = df['Price_LKR'].round(-2) # Round to nearest 100 LKR

# 2. PREPROCESSING
# Calculate Age
current_year = datetime.now().year
df['Age'] = current_year - df['Mfg_Year']

# Drop Mfg_Year as we now have Age
X = df.drop(columns=['Price_LKR', 'Mfg_Year'])
y = df['Price_LKR']

# Convert text categories into numbers (One-Hot Encoding)
X_encoded = pd.get_dummies(X, columns=['Bike_Type', 'Fuel_Type', 'City'])

# Save the exact column names so our future inputs match exactly
training_columns = X_encoded.columns

# 3. TRAIN THE MODEL
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. EVALUATE
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Model Mean Absolute Error: ±{mae:.2f} LKR per day\n")

# 5. TEST A NEW PREDICTION (Simulating a vendor adding a bike)
print("--- Testing Price Suggestion for a New Bike ---")
new_bike = pd.DataFrame([{
    'Engine_CC': 125,
    'Is_Peak_Season': 1, # Vendor is adding this during peak season
    'Age': 2, # 2 years old
    'Bike_Type_Scooter': 1, 'Bike_Type_Gear Bike': 0,
    'Fuel_Type_Petrol': 1, 'Fuel_Type_Electric': 0,
    'City_Colombo': 0, 'City_Deniyaya': 0, 'City_Galle': 0, 'City_Mirissa': 0, 'City_Thalpe': 1
}])

# Ensure the new data has the exact same columns as the training data
new_bike = new_bike.reindex(columns=training_columns, fill_value=0)

suggested_price = model.predict(new_bike)[0]
print(f"Suggested Price for this vendor: {suggested_price:.2f} LKR per day")

# 6. EXPORT THE MODEL
# This saves the model as a file to use in your backend
joblib.dump(model, 'bike_price_model.pkl')
joblib.dump(training_columns, 'model_columns.pkl')
print("\nModel saved successfully!")

Model Mean Absolute Error: ±194.24 LKR per day

--- Testing Price Suggestion for a New Bike ---
Suggested Price for this vendor: 2639.18 LKR per day

Model saved successfully!


In [18]:
print("\n--- Running Scenario Tests ---")

# Define our test scenarios
scenarios = [
    {
        "name": "Premium: Peak Season Thalpe (New Scooter)",
        "data": {'Engine_CC': 110, 'Is_Peak_Season': 1, 'Age': 1,
                 'Bike_Type_Scooter': 1, 'Bike_Type_Gear Bike': 0,
                 'Fuel_Type_Petrol': 1, 'Fuel_Type_Electric': 0,
                 'City_Colombo': 0, 'City_Deniyaya': 0, 'City_Galle': 0, 'City_Mirissa': 0, 'City_Thalpe': 1}
    },
    {
        "name": "Budget: Off-Peak Deniyaya (Old Gear Bike)",
        "data": {'Engine_CC': 150, 'Is_Peak_Season': 1, 'Age': 2,
                 'Bike_Type_Scooter': 0, 'Bike_Type_Gear Bike': 1,
                 'Fuel_Type_Petrol': 1, 'Fuel_Type_Electric': 0,
                'City_Deniyaya': 1}
    },
    {
        "name": "Average: Galle City Commute (Electric)",
        "data": {'Engine_CC': 125, 'Is_Peak_Season': 0, 'Age': 3,
                 'Bike_Type_Scooter': 1, 'Bike_Type_Gear Bike': 0,
                 'Fuel_Type_Petrol': 0, 'Fuel_Type_Electric': 1,
                 'City_Colombo': 0, 'City_Deniyaya': 0, 'City_Galle': 1, 'City_Mirissa': 0, 'City_Thalpe': 0}
    }
]

# Run the tests
for test in scenarios:
    # Convert to DataFrame
    test_df = pd.DataFrame([test["data"]])
    # Ensure columns match the training data exactly
    test_df = test_df.reindex(columns=training_columns, fill_value=0)

    # Predict
    predicted_price = model.predict(test_df)[0]

    print(f"Scenario: {test['name']}")
    print(f"-> Suggested Price: {predicted_price:.2f} LKR / day\n")


--- Running Scenario Tests ---
Scenario: Premium: Peak Season Thalpe (New Scooter)
-> Suggested Price: 2498.70 LKR / day

Scenario: Budget: Off-Peak Deniyaya (Old Gear Bike)
-> Suggested Price: 3473.00 LKR / day

Scenario: Average: Galle City Commute (Electric)
-> Suggested Price: 1861.00 LKR / day



In [20]:
# 6. EXPORT THE MODEL
# This saves the model as a .joblib file to use in your backend
import joblib

joblib.dump(model, 'bike_price_model.joblib')
joblib.dump(training_columns, 'model_columns.joblib')

print("\nModel saved successfully as .joblib!")


Model saved successfully as .joblib!
